In [ ]:
print('===== TORCH BRIDGE V15 (native try/catch + TKVDBG + memory guards, backend a68b783) =====')
print('===== TORCH BRIDGE V13 (device-prep + causal SDPA + fp32 logsoftmax, backend c7920f6) =====')
print('===== TORCH BRIDGE V12 (no diag cells, train cell standalone) =====')
print('===== TORCH BRIDGE V11 (extern C) =====')
print('===== TORCH BRIDGE V10 (data_ptr cast fix) =====')
print('===== TORCH BRIDGE V9 (data_ptr int64 fix) =====')
print('===== TORCH BRIDGE V8 (item<float> fix, backend ec360bb) =====')
print('===== TORCH BRIDGE V7 (backend c2dc54a - clean declarations) =====')
print('===== TORCH BRIDGE V6 (simple incflag) =====')
print('===== TORCH BRIDGE FROM TKV - V2 (Python.h fix) =====')
print('===== GPU BRIDGE SINH TU TOKENVECTOR (.tkv -> .torch.cpp -> .so) =====')
# 1/8 - GPU + deps (real GPU timing on both sides)
import os
os.environ['CUDA_LAUNCH_BLOCKING']='1'
os.environ['TKV_FULL_DEBUG']='1'
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
!nvidia-smi
import sys
print(sys.version)
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!pip -q install transformers

In [ ]:
# 2/8 - Fresh clone TokenVector (bridge source)
import subprocess
d='/content/TokenVector'
subprocess.run('rm -rf '+d, shell=True)
r=subprocess.run('git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
print('clone rc=', r.returncode)
print('HEAD', subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 3/8 - clone CUDA backend (co .tkv + .torch.cpp sinh san) + build AdamW kernel + bridge tu .cpp
import subprocess, os, sys, shutil, ctypes, sysconfig
cud='/content/TokenVectorCuda'
subprocess.run('rm -rf '+cud, shell=True)
subprocess.run('git clone --depth 1 --branch prototype/cuda-backend https://github.com/nguyenhungtran18/TokenVector.git '+cud, shell=True, capture_output=True, text=True, timeout=600)
print('CUDA backend HEAD', subprocess.check_output('git -C '+cud+' rev-parse --short HEAD', shell=True).decode().strip())
nvcc=shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'
# AdamW kernel (.tkv -> .cuda.cu, da co san)
adamw_cu=cud+'/Release/cuda_backend/examples/adamw.cuda.cu'
adamw_so=cud+'/Release/cuda_backend/examples/tkv_cuda_kernels.so'
p0=subprocess.run(nvcc+' -O2 -shared -std=c++20 --expt-relaxed-constexpr -x cu -Xcompiler -fPIC '+adamw_cu+' -o '+adamw_so+' -lcudart', shell=True, capture_output=True, text=True, timeout=600)
print('adamw kernel build rc=',p0.returncode)
assert p0.returncode==0, p0.stderr[:2000]
os.environ['TKV_CUDA_KERNELS_SO']=adamw_so
os.environ['TKV_TORCH_GPU']='1'
# Bridge C++ DUOC SINH BOI TOKENVECTOR
gen=cud+'/Release/cuda_backend/examples/torch_fused_port.torch.cpp'
print('generated bridge exists=',os.path.exists(gen),' bytes=',os.path.getsize(gen) if os.path.exists(gen) else 0)
torch_dir=subprocess.check_output([sys.executable,'-c','import torch,os; print(os.path.dirname(torch.__file__))']).decode().strip()
incs=[torch_dir+'/include',torch_dir+'/include/torch/csrc/api/include']
pyinc=sysconfig.get_paths()['include']
print('python include=',pyinc, os.path.exists(pyinc+'/Python.h'))
incs.append(pyinc)
libdir=torch_dir+'/lib'
so='/content/TokenVector/Release/tools/simd_native/tkv_torch_fused.so'
incflag=''
for xx in incs:
    incflag=incflag+'-I'+xx+' '
cmd=(nvcc+' -O2 -shared -std=c++20 --expt-relaxed-constexpr -x cu -Xcompiler -fPIC '
     +gen+' -o '+so+' '+incflag+'-L'+libdir+' '
     +'-Xlinker -rpath -Xlinker '+libdir+' -ltorch -ltorch_cpu -lc10 -lc10_cuda -ltorch_cuda -lcudart -ldl')
p=subprocess.run(cmd,shell=True,capture_output=True,text=True,timeout=1800)
print('GENERATED BRIDGE BUILD rc=',p.returncode)
if p.returncode!=0: print('STDERR:',p.stderr[:6000])
os.environ['TKV_TORCH_PATH']=so
os.environ['LD_LIBRARY_PATH']=libdir+':'+os.environ.get('LD_LIBRARY_PATH','')
lib=ctypes.CDLL(so)
print('ctypes load OK', os.path.exists(so))

In [ ]:
# 4/8 - Data + pretrained GPT2-medium + identical batches
import numpy as np, torch, urllib.request, copy, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, GPT2Config
T=512; V=50257; NL=24; H=1024; NH=16; HD=64; STEPS=60
tok=AutoTokenizer.from_pretrained('gpt2')
url='https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
text=urllib.request.urlopen(url, timeout=60).read().decode('utf-8')
ids_all=np.array(tok.encode(text), dtype=np.int64)
print('corpus chars=',len(text),' tokens=',len(ids_all))
split=int(len(ids_all)*0.9); train_ids=ids_all[:split]; val_ids=ids_all[split:]
print('train/val tokens=',len(train_ids),len(val_ids))
cfg=GPT2Config(n_layer=NL,n_head=NH,n_embd=H,n_positions=1024,vocab_size=V,
               layer_norm_epsilon=1e-6, activation_function='gelu',
               resid_pdrop=0.0, attn_pdrop=0.0, embd_pdrop=0.0)
model=AutoModelForCausalLM.from_pretrained('gpt2-medium', config=cfg).eval()
sd=model.state_dict()
sd_initial={k:v.detach().clone() for k,v in sd.items()}
def blocks():
    out=[sd['transformer.wte.weight'].numpy(), sd['transformer.wpe.weight'].numpy()[:T],
         sd['transformer.ln_f.weight'].numpy(), sd['transformer.ln_f.bias'].numpy()]
    for i in range(NL):
        p='transformer.h.%d.'%i
        out += [sd[p+'ln_1.weight'].numpy(), sd[p+'ln_1.bias'].numpy(),
                sd[p+'attn.c_attn.weight'].numpy().copy(), sd[p+'attn.c_attn.bias'].numpy(),
                sd[p+'attn.c_proj.weight'].numpy().copy(), sd[p+'attn.c_proj.bias'].numpy(),
                sd[p+'ln_2.weight'].numpy(), sd[p+'ln_2.bias'].numpy(),
                sd[p+'mlp.c_fc.weight'].numpy().copy(), sd[p+'mlp.c_fc.bias'].numpy(),
                sd[p+'mlp.c_proj.weight'].numpy().copy(), sd[p+'mlp.c_proj.bias'].numpy()]
    return out
params=blocks()
flat=np.concatenate([x.reshape(-1) for x in params]).astype(np.float32)
assert len(params)==4+NL*12
rng=np.random.default_rng(7)
batches=[]
for _ in range(STEPS):
    s=int(rng.integers(0,len(train_ids)-T-1))
    batches.append((train_ids[s:s+T].copy(), train_ids[s+1:s+T+1].copy()))
print('tensors=',len(params),' params=',flat.size,' batches=',len(batches))
del sd_initial
gc.collect()
print('freed sd_initial (1.4GB host)')
def flat_to_state(arr, template):
    d=dict(template); off=0
    def take(shape):
        nonlocal off
        z=np.asarray(arr[off:off+int(np.prod(shape))],dtype=np.float32).reshape(shape); off+=z.size
        return torch.from_numpy(z.copy())
    d['transformer.wte.weight']=take((V,H))
    wpe=torch.zeros(1024,H,dtype=torch.float32); wpe[:T]=take((T,H)); d['transformer.wpe.weight']=wpe
    d['transformer.ln_f.weight']=take((H,)); d['transformer.ln_f.bias']=take((H,))
    for i in range(NL):
        p='transformer.h.%d.'%i
        d[p+'ln_1.weight']=take((H,)); d[p+'ln_1.bias']=take((H,))
        d[p+'attn.c_attn.weight']=take((H,3*H)); d[p+'attn.c_attn.bias']=take((3*H,))
        d[p+'attn.c_proj.weight']=take((H,H)); d[p+'attn.c_proj.bias']=take((H,))
        d[p+'ln_2.weight']=take((H,)); d[p+'ln_2.bias']=take((H,))
        d[p+'mlp.c_fc.weight']=take((H,4*H)); d[p+'mlp.c_fc.bias']=take((4*H,))
        d[p+'mlp.c_proj.weight']=take((4*H,H)); d[p+'mlp.c_proj.bias']=take((H,))
    print('consumed',off,'of',arr.size)
    return d



In [ ]:
# 5/8 - PyTorch eager reference (same model/data, real GPU time)
import time, statistics as st
ref_model=copy.deepcopy(model).train().to('cuda')
ref_opt=torch.optim.AdamW(ref_model.parameters(), lr=5e-5, betas=(0.9,0.999), eps=1e-8, weight_decay=0.0)
ref_losses=[]; ref_times=[]
for step,(xb,yb) in enumerate(batches):
    xi=torch.from_numpy(xb.astype(np.int64)).unsqueeze(0).cuda()
    yi=torch.from_numpy(yb.astype(np.int64)).unsqueeze(0).cuda()
    t0=time.perf_counter()
    loss=ref_model(input_ids=xi, labels=yi).loss
    loss.backward(); ref_opt.step(); ref_opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    ref_times.append((time.perf_counter()-t0)*1000); ref_losses.append(float(loss))
    if step%50==0 or step==STEPS-1: print('torch step %4d loss %.4f  %.1f ms'%(step,loss,ref_times[-1]))
print('TORCH median(ms) last50=%.1f  loss %.4f -> %.4f'%(st.median(ref_times[-50:]),ref_losses[0],ref_losses[-1]))
torch_ref_losses=list(ref_losses)
del ref_model, ref_opt, xi, yi; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 6/8 - Register params + optimizer in native fullstep bridge
F,I32,I64=ctypes.c_float, ctypes.c_int32, ctypes.c_int64
lib.TkvFullBegin.argtypes=[I64]*6; lib.TkvFullBegin.restype=ctypes.c_int
lib.TkvFullSetTensor.argtypes=[I64,ctypes.POINTER(F),I64]; lib.TkvFullSetTensor.restype=ctypes.c_int
lib.TkvFullSetIds.argtypes=[ctypes.POINTER(I32),I64]; lib.TkvFullSetIds.restype=ctypes.c_int
lib.TkvFullSetTargets.argtypes=[ctypes.POINTER(I32),I64]; lib.TkvFullSetTargets.restype=ctypes.c_int
lib.TkvFullSetGrad.argtypes=[ctypes.POINTER(F),I64]; lib.TkvFullSetGrad.restype=ctypes.c_int
lib.TkvFullSetOpt.argtypes=[ctypes.POINTER(F)]*3+[F]*5+[I64]*2; lib.TkvFullSetOpt.restype=ctypes.c_int
lib.TkvFullStep124m.argtypes=[I64]; lib.TkvFullStep124m.restype=F
lib.TkvFullGetParams.argtypes=[ctypes.POINTER(F),I64]; lib.TkvFullGetParams.restype=ctypes.c_int
n=flat.size
grad=np.zeros(n,dtype=np.float32); m=np.zeros(n,dtype=np.float32); v=np.zeros(n,dtype=np.float32)
print('BEGIN', lib.TkvFullBegin(len(params),NL,H,NH,HD,V))
for i,a in enumerate(params):
    aa=np.ascontiguousarray(a,dtype=np.float32)
    r=lib.TkvFullSetTensor(i, aa.ctypes.data_as(ctypes.POINTER(F)), aa.size)
    if r!=0: print('SET_FAIL',i,r)
xb0,yb0=batches[0]
print('IDS', lib.TkvFullSetIds(xb0.astype(np.int32).ctypes.data_as(ctypes.POINTER(I32)),T))
print('TGT', lib.TkvFullSetTargets(yb0.astype(np.int32).ctypes.data_as(ctypes.POINTER(I32)),T))
print('GRAD', lib.TkvFullSetGrad(grad.ctypes.data_as(ctypes.POINTER(F)),n))
print('OPT', lib.TkvFullSetOpt(flat.ctypes.data_as(ctypes.POINTER(F)),m.ctypes.data_as(ctypes.POINTER(F)),v.ctypes.data_as(ctypes.POINTER(F)),5e-5,0.9,0.999,1e-8,0.0,STEPS,0))
lib.TkvFullAdamwPath.argtypes=[]; lib.TkvFullAdamwPath.restype=ctypes.c_int
print('ADAMW PATH bind OK')
lib.TkvFullStep124mFwd.argtypes=[I64]; lib.TkvFullStep124mFwd.restype=F
import torch as _t
_free,_tot=_t.cuda.mem_get_info()
print('GPU free/total GB %.2f/%.2f'%(_free/2**30,_tot/2**30))
print('host bufs flat/m/v/grad GB %.2f'%((flat.nbytes+m.nbytes+v.nbytes+grad.nbytes)/2**30))

In [ ]:
# 6a/8 - SMOKE: forward-only (khong prep/backward) - kiem tra loss + VRAM
import torch, traceback
try:
    free,total=torch.cuda.mem_get_info(); print("GPU free truoc fwd %.2f GB"%(free/2**30), flush=True)
    l_fwd=float(lib.TkvFullStep124mFwd(1))
    print("TKV forward-only loss=%.4f   torch step0=%.4f"%(l_fwd, torch_ref_losses[0]), flush=True)
    free,total=torch.cuda.mem_get_info(); print("GPU free sau fwd %.2f GB"%(free/2**30), flush=True)
    if l_fwd < -900: print("*** NATIVE SENTINEL - xem stderr TKV_ERR ***")
except Exception:
    traceback.print_exc()

In [ ]:
# 7/8 - Native fullstep training + head-to-head benchmark
import time, statistics as st, gc, traceback
tkv_losses=[]; tkv_times=[]
try:
    for step,(xb,yb) in enumerate(batches):
        lib.TkvFullSetIds(xb.astype(np.int32).ctypes.data_as(ctypes.POINTER(I32)),T)
        lib.TkvFullSetTargets(yb.astype(np.int32).ctypes.data_as(ctypes.POINTER(I32)),T)
        t1=time.perf_counter()
        loss=float(lib.TkvFullStep124m(0))
        tkv_times.append((time.perf_counter()-t1)*1000); tkv_losses.append(loss)
        if step==0:
            print("step0 loss=%.4f  %.1f ms"%(loss,tkv_times[-1]), flush=True)
            free,total=torch.cuda.mem_get_info(); print("GPU free sau step0 %.2f/%.2f GB"%(free/2**30,total/2**30), flush=True)
            del flat, m, v, grad
            gc.collect()
            print("da giai phong host bufs (flat/m/v/grad)", flush=True)
        if step%10==0 or step==STEPS-1:
            print("tkv  step %4d loss %.4f  %.1f ms"%(step,loss,tkv_times[-1]), flush=True)
        if loss < -900:
            print("*** NATIVE SENTINEL step",step,"- xem stderr TKV_ERR ***", flush=True)
            break
except Exception:
    traceback.print_exc()
if len(tkv_losses)>1:
    n50=min(50,len(tkv_times))
    print("TKV  median(ms) last%d=%.1f  loss %.4f -> %.4f"%(n50,st.median(tkv_times[-n50:]),tkv_losses[0],tkv_losses[-1]))
    print("TORCH median(ms) last50=%.1f  loss %.4f -> %.4f"%(st.median(ref_times[-50:]),torch_ref_losses[0],torch_ref_losses[-1]))
    print("SPEEDUP vs torch = %.2fx"%(st.median(ref_times[-50:])/st.median(tkv_times[-n50:])))
    print("ADAMW PATH (1=TKV .tkv kernel, 2=ATen fallback, 0=never):", lib.TkvFullAdamwPath())
    print("first-step loss: torch %.4f / tkv %.4f (close = layout OK)"%(torch_ref_losses[0],tkv_losses[0]))
    print("BRIDGE = TOKENVECTOR GENERATED C++ ; HEADS:", subprocess.check_output("git -C /content/TokenVector rev-parse --short HEAD",shell=True).decode().strip(), subprocess.check_output("git -C /content/TokenVectorCuda rev-parse --short HEAD",shell=True).decode().strip())
else:
    print("KHONG CO STEP NAO CHAY - xem traceback / stderr o tren")